In [26]:
!pip -q install opencv-python
!pip -q install augly

In [27]:
import os
import sys
import time
import random
from multiprocessing import Process, Queue

import cv2
import numpy as np
from PIL import Image, ImageColor

import tensorflow as tf
import tensorflow.keras.layers as L
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Activation,
    Add,
    Concatenate,
    Conv2D,
    Conv2DTranspose,
    Dropout,
    LayerNormalization,)

import augly.image as imaugs

In [30]:
class Symbols:
    BACKGROUND = [0]
    LEDGERLINE = [2]
    BARLINE_BETWEEN = [3]
    BARLINE_END = [4]
    ALL_BARLINES = BARLINE_BETWEEN + BARLINE_END
    REPEAT_DOTS = [7]
    G_GLEF = [10]
    C_CLEF = [11, 12]  
    F_CLEF = [13]
    ALL_CLEFS = G_GLEF + C_CLEF + F_CLEF
    NUMBERS = [19, 20]  
    TIME_SIGNATURE_SUBSET = [21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 33, 34]  
    TIME_SIGNATURE = TIME_SIGNATURE_SUBSET + [31, 32]  # Oemer hasn't used these in the past
    NOTEHEAD_FULL_ON_LINE = [35]
    UNKNOWN = [36, 38, 40, 128, 143, 144, 148, 150, 157, 159, 160, 161, 162, 163, 164, 167, 170, 171] 
    NOTEHEAD_FULL_BETWEEN_LINES = [37]
    NOTEHEAD_HOLLOW_ON_LINE = [39]
    NOTEHEAD_HOLLOW_BETWEEN_LINE = [41]
    WHOLE_NOTE_ON_LINE = [43]
    WHOLE_NOTE_BETWEEN_LINE = [45]
    DOUBLE_WHOLE_NOTE_ON_LINE = [47]
    DOUBLE_WHOLE_NOTE_BETWEEN_LINE = [49]
    NOTEHEADS_SOLID = NOTEHEAD_FULL_ON_LINE + NOTEHEAD_FULL_BETWEEN_LINES
    NOTEHEADS_HOLLOW = NOTEHEAD_HOLLOW_ON_LINE + NOTEHEAD_HOLLOW_BETWEEN_LINE
    NOTEHEADS_WHOLE = WHOLE_NOTE_ON_LINE + WHOLE_NOTE_BETWEEN_LINE + DOUBLE_WHOLE_NOTE_ON_LINE + DOUBLE_WHOLE_NOTE_BETWEEN_LINE
    NOTEHEADS_ALL = NOTEHEAD_FULL_ON_LINE + NOTEHEAD_FULL_BETWEEN_LINES + NOTEHEAD_HOLLOW_ON_LINE + NOTEHEAD_HOLLOW_BETWEEN_LINE + WHOLE_NOTE_ON_LINE + WHOLE_NOTE_BETWEEN_LINE + DOUBLE_WHOLE_NOTE_ON_LINE + DOUBLE_WHOLE_NOTE_BETWEEN_LINE
    DOT = [51]
    STEM = [52]
    TREMOLO = [53, 54, 55, 56]  
    FLAG_DOWN = [58, 60, 61, 62, 63]  
    FLAG_UP = [64, 66, 67, 68, 69]  
    FLAT = [70]
    NATURAL = [72]
    SHARP = [74]
    DOUBLE_SHARP = [76]
    ALL_ACCIDENTALS = FLAT + NATURAL + SHARP + DOUBLE_SHARP
    KEY_FLAT = [78]
    KEY_NATURAL = [79]
    KEY_SHARP = [80]
    ALL_KEYS = KEY_FLAT + KEY_NATURAL + KEY_SHARP
    ACCENT_ABOVE = [81]
    ACCENT_BELOW = [82]
    STACCATO_ABOVE = [83]
    STACCATO_BELOW = [84]
    TENUTO_ABOVE = [85]
    TENUTO_BELOW = [86]
    STACCATISSIMO_ABOVE = [87]
    STACCATISSIMO_BELOW = [88]
    MARCATO_ABOVE = [89]
    MARCATO_BELOW = [90]
    FERMATA_ABOVE = [91]
    FERMATA_BELOW = [92]
    BREATH_MARK = [93]
    REST_LARGE = [95]
    REST_LONG = [96]
    REST_BREVE = [97]
    REST_FULL = [98]
    REST_QUARTER = [99]
    REST_EIGHTH = [100]
    REST_SIXTEENTH = [101]
    REST_THIRTY_SECOND = [102]
    REST_SIXTY_FOURTH = [103]
    REST_ONE_HUNDRED_TWENTY_EIGHTH = [104]
    ALL_RESTS_EXCEPT_LARGE = REST_LONG + REST_BREVE + REST_FULL + REST_QUARTER + REST_EIGHTH + REST_SIXTEENTH + REST_THIRTY_SECOND + REST_SIXTY_FOURTH + REST_ONE_HUNDRED_TWENTY_EIGHTH
    ALL_RESTS = ALL_RESTS_EXCEPT_LARGE
    TRILL = [127]
    GRUPPETO = [129]
    MORDENT = [130]
    DOWN_BOW = [131]
    UP_BOW = [132]
    SYMBOL = [133, 134, 135, 138, 139, 141, 142]  
    TUPETS = [136, 137, 149, 151, 152, 153, 154, 155, 156]
    SLUR_AND_TIE = [145, 147]
    BEAM = [146]
    STAFF = [165]

DENSE_DATASET_DEFINITIONS = Symbols()

In [29]:
# constant_min.py
DEF = DENSE_DATASET_DEFINITIONS

CLASS_CHANNEL_LIST = [
    DEF.STEM + DEF.ALL_RESTS_EXCEPT_LARGE + DEF.BARLINE_BETWEEN + DEF.BARLINE_END,
    DEF.NOTEHEADS_ALL,
    DEF.ALL_CLEFS + DEF.ALL_KEYS + DEF.ALL_ACCIDENTALS,
]

CLASS_CHANNEL_MAP = {color: idx+1 for idx, colors in enumerate(CLASS_CHANNEL_LIST) for color in colors}

CHANNEL_NUM = len(CLASS_CHANNEL_LIST) + 1  # Plus 'background' and 'others' channel.

In [32]:
# build_label.py
HALF_WHOLE_NOTE = DEF.NOTEHEADS_HOLLOW + DEF.NOTEHEADS_WHOLE + [42]

def fill_hole(gt, tar_color):
    assert tar_color in HALF_WHOLE_NOTE
    tar = np.where(gt==tar_color, 1, 0).astype(np.uint8)
    cnts, _ = cv2.findContours(tar, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in cnts:
        x, y, w, h = cv2.boundingRect(cnt)

        for yi in range(y, y+h):
            cur = x
            cand_y = []
            cand_x = []
            while cur <= x+w:
                if tar[yi, cur] > 0:
                    break
                cur += 1
            while cur <= x+w:
                if tar[yi, cur] == 0:
                    break
                cur += 1
            while cur <= x+w:
                if tar[yi, cur] > 0:
                    break
                cand_y.append(yi)
                cand_x.append(cur)
                cur += 1
            if cur <= x+w:
                cand_y = np.array(cand_y)
                cand_x = np.array(cand_x)
                tar[cand_y, cand_x] = 1

        for xi in range(x, x+w):
            cur = y
            cand_y = []
            cand_x = []
            while cur <= y+h:
                if tar[cur, xi] > 0:
                    break
                cur += 1
            while cur <= y+h:
                if tar[cur, xi] == 0:
                    break
                cur += 1
            while cur <= y+h:
                if tar[cur, xi] > 0:
                    break
                cand_y.append(cur)
                cand_x.append(xi)
                cur += 1
            if cur <= y+h:
                cand_y = np.array(cand_y)
                cand_x = np.array(cand_x)
                tar[cand_y, cand_x] = 1

    return tar


def build_label(seg_path):
    img = Image.open(seg_path)
    arr = np.array(img)
    color_set = set(np.unique(arr))
    color_set.remove(0)  # Remove background color from the candidates

    total_chs = CHANNEL_NUM
    output = np.zeros(arr.shape + (total_chs,))

    output[..., 0] = np.where(arr==0, 1, 0)
    for color in color_set:
        ch = CLASS_CHANNEL_MAP.get(color, 0)
        if (ch != 0) and color in HALF_WHOLE_NOTE:
            note = fill_hole(arr, color)
            output[..., ch] += note
        else:
            output[..., ch] += np.where(arr==color, 1, 0)
    return output


def find_example(dataset_path: str, color: int, max_count=100, mark_value=200):
    files = os.listdir(dataset_path)
    random.shuffle(files)
    for ff in files[:max_count]:
        path = os.path.join(dataset_path, ff)
        img = Image.open(path)
        arr = np.array(img)
        if color in arr:
            return np.where(arr==color, mark_value, arr)

In [33]:
# unet.py
def conv_block(input_tensor, channel, kernel_size, strides=(2, 2), dilation_rate=1, dropout_rate=0.4):
    skip = input_tensor

    input_tensor = LayerNormalization()(Activation("relu")(input_tensor))
    input_tensor = Dropout(dropout_rate)(input_tensor)
    input_tensor = Conv2D(channel, kernel_size, strides=strides, dilation_rate=dilation_rate, padding="same")(input_tensor)

    input_tensor = LayerNormalization()(Activation("relu")(input_tensor))
    input_tensor = Dropout(dropout_rate)(input_tensor)
    input_tensor = Conv2D(channel, kernel_size, strides=(1, 1), dilation_rate=dilation_rate, padding="same")(input_tensor)

    if strides != (1, 1):
        skip = Conv2D(channel, (1, 1), strides=strides, padding="same")(skip)
    input_tensor = Add()([input_tensor, skip])

    return input_tensor


def transpose_conv_block(input_tensor, channel, kernel_size, strides=(2, 2), dropout_rate=0.4):
    skip = input_tensor

    input_tensor = LayerNormalization()(Activation("relu")(input_tensor))
    input_tensor = Dropout(dropout_rate)(input_tensor)
    input_tensor = Conv2D(channel, kernel_size, strides=(1, 1), padding="same")(input_tensor)

    input_tensor = LayerNormalization()(Activation("relu")(input_tensor))
    input_tensor = Dropout(dropout_rate)(input_tensor)
    input_tensor = Conv2DTranspose(channel, kernel_size, strides=strides, padding="same")(input_tensor)

    if strides != (1, 1):
        skip = Conv2DTranspose(channel, (1, 1), strides=strides, padding="same")(skip)
    input_tensor = Add()([input_tensor, skip])

    return input_tensor


def semantic_segmentation(win_size=256, multi_grid_layer_n=1, multi_grid_n=5, out_class=2, dropout=0.4):
    input_score = Input(shape=(win_size, win_size, 3), name="input_score_48")
    en = Conv2D(2**7, (7, 7), strides=(1, 1), padding="same")(input_score)

    en_l1 = conv_block(en, 2**7, (3, 3), strides=(2, 2))
    en_l1 = conv_block(en_l1, 2**7, (3, 3), strides=(1, 1))

    en_l2 = conv_block(en_l1, 2**7, (3, 3), strides=(2, 2))
    en_l2 = conv_block(en_l2, 2**7, (3, 3), strides=(1, 1))
    en_l2 = conv_block(en_l2, 2**7, (3, 3), strides=(1, 1))

    en_l3 = conv_block(en_l2, 2**7, (3, 3), strides=(2, 2))
    en_l3 = conv_block(en_l3, 2**7, (3, 3), strides=(1, 1))
    en_l3 = conv_block(en_l3, 2**7, (3, 3), strides=(1, 1))
    en_l3 = conv_block(en_l3, 2**7, (3, 3), strides=(1, 1))

    en_l4 = conv_block(en_l3, 2**8, (3, 3), strides=(2, 2))
    en_l4 = conv_block(en_l4, 2**8, (3, 3), strides=(1, 1))
    en_l4 = conv_block(en_l4, 2**8, (3, 3), strides=(1, 1))
    en_l4 = conv_block(en_l4, 2**8, (3, 3), strides=(1, 1))
    en_l4 = conv_block(en_l4, 2**8, (3, 3), strides=(1, 1))

    feature = en_l4
    for _ in range(multi_grid_layer_n):
        feature = LayerNormalization()(Activation("relu")(feature))
        feature = Dropout(dropout)(feature)
        m = LayerNormalization()(Conv2D(2**9, (1, 1), strides=(1, 1), padding="same", activation="relu")(feature))
        multi_grid = m
        for ii in range(multi_grid_n):
            m = LayerNormalization()(Conv2D(2**9, (3, 3), strides=(1, 1), dilation_rate=2**ii, padding="same", activation="relu")(feature))

            multi_grid = Concatenate()([multi_grid, m])
        multi_grid = Dropout(dropout)(multi_grid)
        feature = Conv2D(2**9, (1, 1), strides=(1, 1), padding="same")(multi_grid)

    feature = LayerNormalization()(Activation("relu")(feature))

    feature = Conv2D(2**8, (1, 1), strides=(1, 1), padding="same")(feature)
    feature = Add()([feature, en_l4])
    de_l1 = transpose_conv_block(feature, 2**7, (3, 3), strides=(2, 2))

    skip = de_l1
    de_l1 = LayerNormalization()(Activation("relu")(de_l1))
    de_l1 = Concatenate()([de_l1, LayerNormalization()(Activation("relu")(en_l3))])
    de_l1 = Dropout(dropout)(de_l1)
    de_l1 = Conv2D(2**7, (1, 1), strides=(1, 1), padding="same")(de_l1)
    de_l1 = Add()([de_l1, skip])
    de_l2 = transpose_conv_block(de_l1, 2**7, (3, 3), strides=(2, 2))

    skip = de_l2
    de_l2 = LayerNormalization()(Activation("relu")(de_l2))
    de_l2 = Concatenate()([de_l2, LayerNormalization()(Activation("relu")(en_l2))])
    de_l2 = Dropout(dropout)(de_l2)
    de_l2 = Conv2D(2**7, (1, 1), strides=(1, 1), padding="same")(de_l2)
    de_l2 = Add()([de_l2, skip])
    de_l3 = transpose_conv_block(de_l2, 2**7, (3, 3), strides=(2, 2))

    skip = de_l3
    de_l3 = LayerNormalization()(Activation("relu")(de_l3))
    de_l3 = Concatenate()([de_l3, LayerNormalization()(Activation("relu")(en_l1))])
    de_l3 = Dropout(dropout)(de_l3)
    de_l3 = Conv2D(2**7, (1, 1), strides=(1, 1), padding="same")(de_l3)
    de_l3 = Add()([de_l3, skip])
    de_l4 = transpose_conv_block(de_l3, 2**7, (3, 3), strides=(2, 2))

    de_l4 = LayerNormalization()(Activation("relu")(de_l4))
    de_l4 = Dropout(dropout)(de_l4)
    out = Conv2D(out_class, (1, 1), strides=(1, 1), activation='softmax', padding="same", name="prediction")(de_l4)

    return Model(inputs=input_score, outputs=out)


# def my_conv_block(inp, kernels, kernel_size=(3, 3), strides=(1, 1)):
#     inp = L.Conv2D(kernels, kernel_size, strides=strides, padding='same', dtype=tf.float32)(inp)
#     out = L.Activation("relu")(L.LayerNormalization()(inp))
#     out = L.SeparableConv2D(kernels, kernel_size, padding='same', dtype=tf.float32)(out)
#     out = L.Activation("relu")(L.LayerNormalization()(out))
#     out = L.Dropout(0.3)(out)
#     out = L.Add()([inp, out])
#     out = L.Activation("relu")(L.LayerNormalization()(out))
#     return out


def my_conv_small_block(inp, kernels, kernel_size=(3, 3), strides=(1, 1)):
    inp = L.Conv2D(kernels, kernel_size, strides=strides, padding='same', dtype=tf.float32)(inp)
    out = L.Activation("relu")(L.LayerNormalization()(inp))
    out = L.Dropout(0.3)(out)
    out = L.Add()([inp, out])
    out = L.Activation("relu")(L.LayerNormalization()(out))
    return out


def my_trans_conv_block(inp, kernels, kernel_size=(3, 3), strides=(1, 1)):
    inp = L.Conv2DTranspose(kernels, kernel_size, strides=strides, padding='same', dtype=tf.float32)(inp)
    out = L.Conv2D(kernels, kernel_size, padding='same', dtype=tf.float32)(inp)
    out = L.Activation("relu")(L.LayerNormalization()(out))
    out = L.Dropout(0.3)(out)
    out = L.Add()([inp, out])
    out = L.Activation("relu")(L.LayerNormalization()(out))
    return out


def u_net(win_size=288, out_class= 3):
    inp = L.Input(shape=(win_size, win_size, 3))
    tensor = L.SeparableConv2D(128, (3, 3), activation="relu", padding='same')(inp)

    l1 = my_conv_small_block(tensor, 64, (3, 3), strides=(2, 2))
    l1 = my_conv_small_block(l1, 64, (3, 3))
    l1 = my_conv_small_block(l1, 64, (3, 3))

    skip = my_conv_small_block(l1, 128, (3, 3), strides=(2, 2))
    l2 = my_conv_small_block(skip, 128, (3, 3))
    l2 = my_conv_small_block(l2, 128, (3, 3))
    l2 = my_conv_small_block(l2, 128, (3, 3))
    l2 = my_conv_small_block(l2, 128, (3, 3))
    l2 = L.Concatenate()([skip, l2])

    l3 = my_conv_small_block(l2, 256, (3, 3))
    l3 = my_conv_small_block(l3, 256, (3, 3))
    l3 = my_conv_small_block(l3, 256, (3, 3))
    l3 = my_conv_small_block(l3, 256, (3, 3))
    l3 = my_conv_small_block(l3, 256, (3, 3))
    l3 = L.Concatenate()([l2, l3])

    bot = my_conv_small_block(l3, 256, (3, 3), strides=(2, 2))  # 32
    st1 = L.SeparableConv2D(256, (3, 3), padding='same', dtype=tf.float32)(bot)
    st1 = L.Activation("relu")(L.LayerNormalization()(st1))
    st2 = L.SeparableConv2D(256, (3, 3), dilation_rate=(2, 2), padding='same', dtype=tf.float32)(bot)
    st2 = L.Activation("relu")(L.LayerNormalization()(st2))
    st3 = L.SeparableConv2D(256, (3, 3), dilation_rate=(6, 6), padding='same', dtype=tf.float32)(bot)
    st3 = L.Activation("relu")(L.LayerNormalization()(st3))
    st4 = L.SeparableConv2D(256, (3, 3), dilation_rate=(12, 12), padding='same', dtype=tf.float32)(bot)
    st4 = L.Activation("relu")(L.LayerNormalization()(st4))
    st = L.Concatenate()([st1, st2, st3, st4])
    st = L.Conv2D(256, (1, 1), padding='same', dtype=tf.float32)(st)
    norm = L.Activation("relu")(L.LayerNormalization()(st))
    bot = my_trans_conv_block(norm, 256, (3, 3), strides=(2, 2))  # 64

    tl3 = L.Conv2D(128, (3, 3), padding='same', dtype=tf.float32)(bot)
    tl3 = L.Activation("relu")(L.LayerNormalization()(tl3))
    tl3 = L.Concatenate()([tl3, l3])
    tl3 = my_conv_small_block(tl3, 128, (3, 3))
    tl3 = my_trans_conv_block(tl3, 128, (3, 3))

    # Head 1
    tl2 = L.Conv2D(128, (3, 3), padding='same', dtype=tf.float32)(tl3)
    tl2 = L.Activation("relu")(L.LayerNormalization()(tl2))
    tl2 = L.Concatenate()([tl2, l2])
    tl2 = my_conv_small_block(tl2, 128, (3, 3))
    tl2 = my_trans_conv_block(tl2, 128, (3, 3), strides=(2, 2))  # 128

    tl1 = L.Conv2D(128, (3, 3), padding='same', dtype=tf.float32)(tl2)
    tl1 = L.Activation("relu")(L.LayerNormalization()(tl1))
    tl1 = L.Concatenate()([tl1, l1])
    tl1 = my_conv_small_block(tl1, 128, (3, 3))
    tl1 = my_trans_conv_block(tl1, 128, (3, 3), strides=(2, 2))  # 256

    out1 = L.Conv2D(out_class, (1, 1), activation='softmax', padding='same', dtype=tf.float32)(tl1)

    return tf.keras.Model(inputs=inp, outputs=out1)

In [34]:
# bbox.py
from typing import Union, Any, List, Tuple, Dict

from cv2.typing import RotatedRect
from numpy import ndarray
from sklearn.cluster import AgglomerativeClustering

BBox = Tuple[int, int, int, int]

def get_bbox(data: ndarray) -> List[BBox]:
    contours, _ = cv2.findContours(data.astype(np.uint8), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        box = (x, y, x+w, y+h)
        bboxes.append(box)
    return bboxes


def get_center(bbox: Union[BBox, ndarray]) -> Tuple[int, int]:
    cen_y = int(round((bbox[1] + bbox[3]) / 2))
    cen_x = int(round((bbox[0] + bbox[2]) / 2))
    return cen_x, cen_y


def get_edge(data):
    if len(data.shape) == 3:
        data = cv2.cvtColor(data, cv2.COLOR_BGR2GRAY)
        data = cv2.GaussianBlur(data, (5, 5), 0)
    data = cv2.Canny(data, 10, 80)
    return data


def merge_nearby_bbox(bboxes: List[BBox], distance: float, x_factor: int = 1, y_factor: int = 1) -> List[BBox]:
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=distance, compute_full_tree=True)
    centers = np.array([(bb[0]+bb[2], bb[1]+bb[3]) for bb in bboxes]) / 2
    centers[:, 0] *= x_factor  # Increase/decrease the x distance
    centers[:, 1] *= y_factor  # Increase/decrease the y distance
    model.fit(centers)
    labels = np.unique(model.labels_)
    new_box = []
    for label in labels:
        idx = np.where(model.labels_ == label)[0]
        xs = [[bboxes[i][0], bboxes[i][2]] for i in idx]
        ys = [[bboxes[i][1], bboxes[i][3]] for i in idx]
        x1, x2 = np.min(xs), np.max(xs)
        y1, y2 = np.min(ys), np.max(ys)
        box = (x1, y1, x2, y2)
        new_box.append(box)
    return new_box


def rm_merge_overlap_bbox(bboxes: List[BBox], mode: str = 'remove', overlap_ratio: float = 0.5) -> List[BBox]:
    assert mode in ['remove', 'merge'], mode

    pts = np.array([(box[2], box[3]) for box in bboxes])
    max_x, max_y = np.max(pts[:, 0]), np.max(pts[:, 1])
    mask = np.zeros((max_y, max_x), dtype=np.uint16)

    box_infos: List[Dict[str, Any]] = []
    for box in bboxes:
        area_size = (box[3]-box[1]) * (box[2]-box[0])
        box_infos.append({"bbox": box, "area_size": area_size})
    box_infos = sorted(box_infos, key=lambda info: info["area_size"], reverse=True)

    records = {}
    for idx, box_info in enumerate(box_infos):
        box = box_info["bbox"]
        region = mask[box[1]:box[3], box[0]:box[2]]
        vals = set(np.unique(region))
        if 0 in vals:
            vals.remove(0)

        if len(vals) == 0:
            records[idx+1] = box_info
            mask[box[1]:box[3], box[0]:box[2]] = idx + 1
            continue

        area_size = box_info["area_size"]
        valid = True
        for val in vals:
            tar_info = records[val]
            tar_size = tar_info["area_size"]
            overlap_size = region[region==val].size
            assert tar_size >= area_size, f"{tar_size}, {area_size}, {val}"
            ratio = overlap_size / area_size
            if ratio > overlap_ratio:
                if mode == "merge":
                    box = box_info['bbox']
                    tar_box = tar_info['bbox']
                    top_x = min(box[0], tar_box[0])
                    top_y = min(box[1], tar_box[1])
                    bt_x = max(box[2], tar_box[2])
                    bt_y = max(box[3], tar_box[3])
                    records[val]['bbox'] = (top_x, top_y, bt_x, bt_y)
                valid = False
                break

        if valid:
            records[idx+1] = box_info
            mask[box[1]:box[3], box[0]:box[2]] = idx + 1

    valid_box = [info['bbox'] for info in records.values()]
    return valid_box


def find_lines(data: ndarray, min_len: int = 10, max_gap: int = 20) -> List[BBox]:
    assert len(data.shape) == 2, f"{type(data)} {data.shape}"

    lines = cv2.HoughLinesP(data.astype(np.uint8), 1, np.pi/180, 50, None, min_len, max_gap)
    new_line = []
    if lines is not None:
        for line in lines:
            line = line[0]
            top_x, bt_x = (line[0], line[2]) if line[0] < line[2] else (line[2], line[0])
            top_y, bt_y = (line[1], line[3]) if line[1] < line[3] else (line[3], line[1])
            new_line.append((top_x, top_y, bt_x, bt_y))
    return new_line


def draw_lines(lines: ndarray, ori_img: ndarray, width: int = 3) -> ndarray:
    img = ori_img.copy()
    for line in lines:
        cv2.line(img, (line[0], line[1]), (line[2], line[3]), (0, 255, 0), width, cv2.LINE_AA)
    return img


def to_rgb_img(data: ndarray) -> ndarray:
    if len(data.shape) >=3:
        return data

    img = np.ones(data.shape + (3,), dtype=np.uint8) * 255
    idx = np.where(data > 0)
    img[idx[0], idx[1]] = 0
    return img


def draw_bounding_boxes(bboxes: List[BBox], img: ndarray, color: Tuple[int, int, int] = (0, 255, 0), width: int = 2, inplace: bool = False) -> ndarray:
    if len(img.shape) < 3:
        img = to_rgb_img(img)
    if not inplace:
        img = np.array(img)
    for (x1, y1, x2, y2) in bboxes:
        cv2.rectangle(img, (x1, y1), (x2, y2), color, width)
    return img


def get_rotated_bbox(data: ndarray) -> List[RotatedRect]:
    contours, _ = cv2.findContours(data.astype(np.uint8), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in contours:
        bboxes.append(cv2.minAreaRect(cnt))
    return bboxes


def draw_rotated_bounding_boxes(bboxes, img, color=(0, 255, 0), width=2, inplace=False):
    if len(img.shape) < 3:
        img = to_rgb_img(img)
    if not inplace:
        img = np.array(img)
    for rbox in bboxes:
        box = cv2.boxPoints(rbox).astype(np.int64)
        cv2.drawContours(img, [box], 0, color, width)
    return img

In [35]:
# # classifier.py
# import pickle
# from pathlib import Path
# from PIL import Image

# from sklearn import svm

# TARGET_WIDTH = 40
# TARGET_HEIGHT = 70
# DISTANCE = 10

# DATASET_PATH = "/kaggle/input/datasets/nguyncaonam/ds2-dense/ds2_dense/segmentation"


# def _collect(color, out_path, samples=100):
#     out_path = Path(out_path)
#     if not out_path.exists():
#         out_path.mkdir(parents=True, exist_ok=True)

#     cur_samples = 0
#     add_space = 10
#     idx = 0
#     while cur_samples < samples:
#         arr = find_example(DATASET_PATH, color)
#         if arr is None:
#             continue
#         arr[arr!=200] = 0
#         boxes = get_bbox(arr)
#         if len(boxes) > 1:
#             boxes = merge_nearby_bbox(boxes, DISTANCE)
#         boxes = rm_merge_overlap_bbox(boxes)
#         for box in boxes:
#             if idx >= samples:
#                 break
#             print(f"{idx+1}/{samples}", end='\r')
#             patch = arr[box[1]-add_space:box[3]+add_space, box[0]-add_space:box[2]+add_space]
#             ratio = random.choice(np.arange(0.6, 1.3, 0.1))
#             tar_w = int(ratio * patch.shape[1])
#             tar_h = int(ratio * patch.shape[0])
#             img = imaugs.resize(Image.fromarray(patch.astype(np.uint8)), width=tar_w, height=tar_h)

#             seed = random.randint(0, 1000)
#             np.float = float  # Monkey patch to workaround removal of np.float
#             img = imaugs.perspective_transform(img, seed=seed, sigma=3)
#             img = np.where(np.array(img)>0, 255, 0)
#             Image.fromarray(img.astype(np.uint8)).save(out_path / f"{idx}.png")
#             idx += 1

#         cur_samples += len(boxes)
#     print()


# def collect_data(samples=400):
#     COLOR_MAP = {
#         74: "sharp",
#         70: "flat",
#         72: "natural",
#         97: 'rest_whole',
#         98: 'rest_half',
#         99: 'rest_quarter',
#         100: 'rest_8th',
#         101: 'rest_16th',
#         102: 'rest_32nd',
#         103: 'rest_64th',
#         10: 'gclef',
#         (11, 12): 'cclef',
#         13: 'fclef',}

#     for color, name in COLOR_MAP.items():
#         print('Current', name)
#         _collect(color, f"train_data/{name}", samples=samples)
#         _collect(color, f"test_data/{name}", samples=samples)


# def train(folders):
#     class_map = {idx: Path(ff).name for idx, ff in enumerate(folders)}
#     train_x = []
#     train_y = []
#     samples = None
#     print("Loading data")
#     for cidx, folder in enumerate(folders):
#         folder = Path(folder)
#         idx = 0
#         for ff in folder.glob('*.png'):
#             if samples is not None and idx >= samples:
#                 break
#             img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
#             arr = np.array(img).flatten()
#             train_x.append(arr)
#             train_y.append(cidx)
#             idx += 1

#     print("Train model")
#     model = svm.SVC()#C=0.1, gamma=0.0001, kernel='poly', degree=2, decision_function_shape='ovo')
#     model.fit(train_x, train_y)
#     return model, class_map

# def build_class_map(folders):
#     return {idx: Path(ff).name for idx, ff in enumerate(folders)}

# def train_tf(folders):
#     import tensorflow as tf
#     class_map = build_class_map(folders)
#     train_x = []
#     train_y = []
#     samples = None
#     print("Loading data")
#     for cidx, folder in enumerate(folders):
#         folder = Path(folder)
#         idx = 0
#         for ff in folder.iterdir():
#             if samples is not None and idx >= samples:
#                 break
#             img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
#             arr = np.array(img)
#             train_x.append(arr)
#             train_y.append(cidx)
#             idx += 1
#     train_x = np.array(train_x)[..., np.newaxis]
#     train_y = tf.one_hot(train_y, len(folders))
#     output_types = (tf.uint8, tf.uint8)
#     output_shapes = ((TARGET_HEIGHT, TARGET_WIDTH, 1), (len(folders)))
#     dataset = tf.data.Dataset.from_generator(lambda: zip(train_x, train_y), output_types=output_types, output_shapes=output_shapes)
#     dataset = dataset.shuffle(len(train_y), reshuffle_each_iteration=True)
#     dataset = dataset.repeat(5)
#     dataset = dataset.batch(16)

#     model = tf.keras.models.Sequential([
#         tf.keras.layers.InputLayer(input_shape=(TARGET_HEIGHT, TARGET_WIDTH, 1)),
#         tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
#         tf.keras.layers.BatchNormalization(),
#         tf.keras.layers.Dropout(0.2),
#         tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
#         tf.keras.layers.BatchNormalization(),
#         tf.keras.layers.Dropout(0.2),
#         tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
#         tf.keras.layers.BatchNormalization(),
#         tf.keras.layers.Dropout(0.2),
#         tf.keras.layers.Flatten(),
#         tf.keras.layers.Dense(128, activation='relu'),
#         tf.keras.layers.Dropout(0.2),
#         tf.keras.layers.Dense(len(folders), activation='softmax')])

#     model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
#     model.fit(dataset, batch_size=16, epochs=10)
#     return model, class_map


# def test(model, folders):
#     test_x = []
#     test_y = []
#     samples = 100
#     print("Loading data")
#     for cidx, folder in enumerate(folders):
#         folder = Path(folder)
#         idx = 0
#         files = list(folder.glob('*.png'))
#         random.shuffle(files)
#         for ff in files:
#             if idx >= samples:
#                 break
#             img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
#             arr = np.array(img).flatten()
#             test_x.append(arr)
#             test_y.append(cidx)
#             idx += 1

#     pred_y = model.predict(test_x)
#     tp_idx = (pred_y == test_y)
#     tp = len(pred_y[tp_idx])
#     acc = tp / len(test_y)
#     print("Accuracy: ", acc)


# def test_tf(model, folders):
#     test_x = []
#     test_y = []
#     print("Loading data")
#     for cidx, folder in enumerate(folders):
#         folder = Path(folder)
#         files = list(folder.iterdir())
#         random.shuffle(files)
#         for ff in files:
#             img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
#             arr = np.array(img)
#             test_x.append(arr)
#             test_y.append(cidx)

#     test_x = np.array(test_x)[..., np.newaxis]
#     test_y = np.array(test_y)
#     test_result = []
#     batch_size = 16
#     for idx in range(0, len(test_x), batch_size):
#         data = test_x[idx:idx+batch_size]
#         pred = model.predict(data)
#         pidx = np.argmax(pred, axis=-1)
#         test_result.extend(list(pidx))

#     test_result = np.array(test_result)
#     tp = test_result[test_result==test_y].size
#     acc = tp / len(test_y)
#     print("Accuracy: ", acc)


# def predict(region, model_name):
#     if np.max(region) == 1:
#         region *= 255
#     m_info = pickle.load(open(f"sklearn_models/{model_name}.model", "rb"))
#     model = m_info['model']
#     w = m_info['w']
#     h = m_info['h']
#     region = Image.fromarray(region.astype(np.uint8)).resize((w, h))
#     pred = model.predict(np.array(region).reshape(1, -1))
#     return m_info['class_map'][pred[0]]

# def train_rests_above8(filename = "rests_above8.model"):
#     folders = ["rest_8th", "rest_16th", "rest_32nd", "rest_64th"]
#     model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
#     test_tf(model, [f"test_data/{folder}" for folder in folders])
#     output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
#     pickle.dump(output, open(filename, "wb"))


# def train_rests(filename = "rests.model"):
#     folders = ["rest_whole", "rest_quarter", "rest_8th"]
#     model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
#     test_tf(model, [f"test_data/{folder}" for folder in folders])
#     output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
#     pickle.dump(output, open(filename, "wb"))


# def train_all_rests(filename = "all_rests.model"):
#     folders = ["rest_whole", "rest_quarter", "rest_8th", "rest_16th", "rest_32nd", "rest_64th"]
#     model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
#     test_tf(model, [f"test_data/{folder}" for folder in folders])
#     output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
#     pickle.dump(output, open(filename, "wb"))


# def train_sfn(filename = "sfn.model"):
#     folders = ["sharp", "flat", "natural"]
#     model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
#     test_tf(model, [f"test_data/{folder}" for folder in folders])
#     output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
#     pickle.dump(output, open(filename, "wb"))


# def train_clefs(filename = "clef.model"):
#     folders = ["gclef", "fclef", "cclef"]
#     model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
#     test_tf(model, [f"test_data/{folder}" for folder in folders])
#     output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
#     pickle.dump(output, open(filename, "wb"))


# def train_noteheads():
#     folders = ["notehead_solid", "notehead_hollow"]
#     model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
#     test_tf(model, [f"test_data/{folder}" for folder in folders])
#     output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
#     pickle.dump(output, open(f"notehead.model", "wb"))

In [36]:
# classifier.py
import pickle
from pathlib import Path
from PIL import Image

from sklearn import svm

TARGET_WIDTH = 40
TARGET_HEIGHT = 70
DISTANCE = 10

DISTANCE_MAP = {
    "gclef": 10,
    "cclef": 60,
    "fclef": 40,
    "sharp": 10,
    "flat": 10,
    "natural": 10,
    "rest_whole": 10,
    "rest_half": 10,
    "rest_quarter": 10,
    "rest_8th": 10,
    "rest_16th": 10,
    "rest_32nd": 10,
    "rest_64th": 10,
}

DATASET_PATH = "/kaggle/input/datasets/nguyncaonam/ds2-dense/ds2_dense/segmentation"

def _collect(colors, out_path, samples=100, merge_distance=None):
    if isinstance(colors, int):
        colors = [colors]

    # Dùng DISTANCE mặc định nếu không truyền merge_distance
    if merge_distance is None:
        merge_distance = DISTANCE

    out_path = Path(out_path)
    if not out_path.exists():
        out_path.mkdir(parents=True, exist_ok=True)

    cur_samples = 0
    add_space = 10
    idx = len(list(out_path.glob("*.png")))

    while cur_samples < samples:
        color = random.choice(colors)
        arr = find_example(DATASET_PATH, color)
        if arr is None:
            continue

        arr[arr != 200] = 0
        boxes = get_bbox(arr)
        if len(boxes) > 1:
            # [FIX 2a] Dùng merge_distance thay vì DISTANCE cố định
            boxes = merge_nearby_bbox(boxes, merge_distance)
        boxes = rm_merge_overlap_bbox(boxes)

        for box in boxes:
            if cur_samples >= samples:
                break
            print(f"{cur_samples+1}/{samples}", end='\r')

            x1 = max(0, box[0] - add_space)
            y1 = max(0, box[1] - add_space)
            x2 = min(arr.shape[1], box[2] + add_space)
            y2 = min(arr.shape[0], box[3] + add_space)

            patch = arr[y1:y2, x1:x2]
            if patch.size == 0:
                continue

            # [FIX 2b] Lọc mẫu rác: bỏ qua patch có quá ít pixel trắng
            # hoặc aspect ratio quá hẹp (thường là đường thẳng bị tách rời)
            white_ratio = np.sum(patch > 0) / patch.size
            patch_h, patch_w = patch.shape[:2]
            aspect_ratio = patch_w / patch_h if patch_h > 0 else 0
            if white_ratio < 0.05 or aspect_ratio < 0.15:
                continue

            ratio = random.choice(np.arange(0.6, 1.3, 0.1))
            tar_w = max(1, int(ratio * patch.shape[1]))
            tar_h = max(1, int(ratio * patch.shape[0]))
            img = imaugs.resize(Image.fromarray(patch.astype(np.uint8)), width=tar_w, height=tar_h)

            seed = random.randint(0, 1000)
            np.float = float  # Monkey patch to workaround removal of np.float
            img = imaugs.perspective_transform(img, seed=seed, sigma=3)
            img = np.where(np.array(img) > 0, 255, 0)
            Image.fromarray(img.astype(np.uint8)).save(out_path / f"{idx}.png")
            idx += 1
            cur_samples += 1

    print()


def collect_data(samples=400):
    color_map = {
        74: "sharp",
        70: "flat",
        72: "natural",
        97: "rest_whole",
        98: "rest_half",
        99: "rest_quarter",
        100: "rest_8th",
        101: "rest_16th",
        102: "rest_32nd",
        103: "rest_64th",
        (11, 12): "cclef",
        10: "gclef",
        13: "fclef",
    }

    for color, name in color_map.items():
        print("Current", name)
        merge_distance = DISTANCE_MAP.get(name, DISTANCE)
        _collect(color, f"train_data/{name}", samples=samples, merge_distance=merge_distance)
        _collect(color, f"test_data/{name}", samples=samples, merge_distance=merge_distance)


def train(folders):
    class_map = {idx: Path(ff).name for idx, ff in enumerate(folders)}
    train_x = []
    train_y = []
    samples = None
    print("Loading data")
    for cidx, folder in enumerate(folders):
        folder = Path(folder)
        idx = 0
        for ff in folder.glob('*.png'):
            if samples is not None and idx >= samples:
                break
            img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
            arr = np.array(img).flatten()
            train_x.append(arr)
            train_y.append(cidx)
            idx += 1

    print("Train model")
    model = svm.SVC()#C=0.1, gamma=0.0001, kernel='poly', degree=2, decision_function_shape='ovo')
    model.fit(train_x, train_y)
    return model, class_map

def build_class_map(folders):
    return {idx: Path(ff).name for idx, ff in enumerate(folders)}

def train_tf(folders):
    import tensorflow as tf
    class_map = build_class_map(folders)
    train_x = []
    train_y = []
    samples = None
    print("Loading data")
    for cidx, folder in enumerate(folders):
        folder = Path(folder)
        idx = 0
        for ff in folder.iterdir():
            if samples is not None and idx >= samples:
                break
            img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
            arr = np.array(img)
            train_x.append(arr)
            train_y.append(cidx)
            idx += 1
    train_x = np.array(train_x)[..., np.newaxis]
    train_y = tf.one_hot(train_y, len(folders))
    output_types = (tf.uint8, tf.uint8)
    output_shapes = ((TARGET_HEIGHT, TARGET_WIDTH, 1), (len(folders)))
    dataset = tf.data.Dataset.from_generator(lambda: zip(train_x, train_y), output_types=output_types, output_shapes=output_shapes)
    dataset = dataset.shuffle(len(train_y), reshuffle_each_iteration=True)
    dataset = dataset.repeat(5)
    dataset = dataset.batch(16)

    model = tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(TARGET_HEIGHT, TARGET_WIDTH, 1)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(len(folders), activation='softmax')])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(dataset, batch_size=16, epochs=10)
    return model, class_map


def test(model, folders):
    test_x = []
    test_y = []
    samples = 100
    print("Loading data")
    for cidx, folder in enumerate(folders):
        folder = Path(folder)
        idx = 0
        files = list(folder.glob('*.png'))
        random.shuffle(files)
        for ff in files:
            if idx >= samples:
                break
            img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
            arr = np.array(img).flatten()
            test_x.append(arr)
            test_y.append(cidx)
            idx += 1

    pred_y = model.predict(test_x)
    tp_idx = (pred_y == test_y)
    tp = len(pred_y[tp_idx])
    acc = tp / len(test_y)
    print("Accuracy: ", acc)


def test_tf(model, folders):
    test_x = []
    test_y = []
    print("Loading data")
    for cidx, folder in enumerate(folders):
        folder = Path(folder)
        files = list(folder.iterdir())
        random.shuffle(files)
        for ff in files:
            img = Image.open(ff).resize((TARGET_WIDTH, TARGET_HEIGHT))
            arr = np.array(img)
            test_x.append(arr)
            test_y.append(cidx)

    test_x = np.array(test_x)[..., np.newaxis]
    test_y = np.array(test_y)
    test_result = []
    batch_size = 16
    for idx in range(0, len(test_x), batch_size):
        data = test_x[idx:idx+batch_size]
        pred = model.predict(data)
        pidx = np.argmax(pred, axis=-1)
        test_result.extend(list(pidx))

    test_result = np.array(test_result)
    tp = test_result[test_result==test_y].size
    acc = tp / len(test_y)
    print("Accuracy: ", acc)


def predict(region, model_name):
    if np.max(region) == 1:
        region *= 255
    m_info = pickle.load(open(f"sklearn_models/{model_name}.model", "rb"))
    model = m_info['model']
    w = m_info['w']
    h = m_info['h']
    region = Image.fromarray(region.astype(np.uint8)).resize((w, h))
    pred = model.predict(np.array(region).reshape(1, -1))
    return m_info['class_map'][pred[0]]

def train_rests_above8(filename = "rests_above8.model"):
    folders = ["rest_8th", "rest_16th", "rest_32nd", "rest_64th"]
    model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
    test_tf(model, [f"test_data/{folder}" for folder in folders])
    output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
    pickle.dump(output, open(filename, "wb"))


def train_rests(filename = "rests.model"):
    folders = ["rest_whole", "rest_quarter", "rest_8th"]
    model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
    test_tf(model, [f"test_data/{folder}" for folder in folders])
    output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
    pickle.dump(output, open(filename, "wb"))


def train_all_rests(filename = "all_rests.model"):
    folders = ["rest_whole", "rest_quarter", "rest_8th", "rest_16th", "rest_32nd", "rest_64th"]
    model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
    test_tf(model, [f"test_data/{folder}" for folder in folders])
    output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
    pickle.dump(output, open(filename, "wb"))


def train_sfn(filename = "sfn.model"):
    folders = ["sharp", "flat", "natural"]
    model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
    test_tf(model, [f"test_data/{folder}" for folder in folders])
    output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
    pickle.dump(output, open(filename, "wb"))


def train_clefs(filename = "clef.model"):
    folders = ["gclef", "fclef", "cclef"]
    model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
    test_tf(model, [f"test_data/{folder}" for folder in folders])
    output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
    pickle.dump(output, open(filename, "wb"))


def train_noteheads():
    folders = ["notehead_solid", "notehead_hollow"]
    model, class_map = train_tf([f"train_data/{folder}" for folder in folders])
    test_tf(model, [f"test_data/{folder}" for folder in folders])
    output = {'model': model, 'w': TARGET_WIDTH, 'h': TARGET_HEIGHT, 'class_map': class_map}
    pickle.dump(output, open(f"notehead.model", "wb"))

In [37]:
# train.py
def get_cvc_data_paths(dataset_path):
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"{dataset_path} not found, download the dataset first.")

    dirs = ["curvature", "ideal", "interrupted", "kanungo", "rotated", "staffline-thickness-variation-v1",
            "staffline-thickness-variation-v2", "staffline-y-variation-v1", "staffline-y-variation-v2",
            "thickness-ratio", "typeset-emulation", "whitespeckles"]

    data = []
    for dd in dirs:
        dir_path = os.path.join(dataset_path, dd)
        folders = os.listdir(dir_path)
        for folder in folders:
            data_path = os.path.join(dir_path, folder)
            imgs = os.listdir(os.path.join(data_path, "image"))
            for img in imgs:
                img_path = os.path.join(data_path, "image", img)
                staffline = os.path.join(data_path, "gt", img)
                symbol_path = os.path.join(data_path, "symbol", img)
                data.append([img_path, staffline, symbol_path])

    return data


def get_deep_score_data_paths(dataset_path):
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"{dataset_path} not found, download the dataset first.")

    imgs = os.listdir(os.path.join(dataset_path, "images"))
    paths = []
    for img in imgs:
        image_path = os.path.join(dataset_path, "images", img)
        seg_path = os.path.join(dataset_path, "segmentation", img.replace(".png", "_seg.png"))
        paths.append((image_path, seg_path))
    return paths


def preprocess_image(img_path):
    image = Image.open(img_path).convert("1")
    params = {}

    if image.mode == "1":
        arr = np.array(image)
        out = np.zeros(arr.shape + (3,), dtype=np.uint8)
        bg_is_white = np.count_nonzero(arr) > (arr.size * 0.7)
        bg_idx = np.where(arr==bg_is_white)

        hue = random.randint(19, 60)
        sat = random.randint(0, 15)
        val = random.randint(70, 100)
        color = ImageColor.getrgb(f"hsv({hue}, {sat}%, {val}%)")
        out[bg_idx[0], bg_idx[1]] = color
        image = Image.fromarray(out)
        params['bg_color'] = {'hue': hue, 'saturation': sat, 'value': val}

    bright = (7 + random.randint(0, 6)) / 10  # 0.7~1.3
    saturation = (5 + random.randint(0, 7)) / 10  # 0.5~1.2
    contrast = (5 + random.randint(0, 10)) / 10  # 0.5~1.5
    aug_image = imaugs.color_jitter(
        image, brightness_factor=bright, saturation_factor=saturation, contrast_factor=contrast)
    params['color_jitter'] = {'brightness': bright, 'saturation': saturation, 'contrast': contrast}

    rad = random.choice(np.arange(0.0001, 2.1, 0.5))
    aug_image = imaugs.blur(aug_image, radius=rad)
    params['blur_radius'] = rad

    factor = random.choice(np.arange(0.0001, 0.26, 0.05))
    aug_image = imaugs.shuffle_pixels(aug_image, factor=factor)
    params['pixel_shuffle_factor'] = factor

    qa = random.randint(0, 100)
    aug_image = imaugs.encoding_quality(aug_image, quality=qa)
    params['image_quality'] = qa

    rat = random.randint(3, 10) / 10
    aug_image = imaugs.pixelization(aug_image, ratio=rat)
    params['pixelize_ratio'] = rat

    return aug_image, params


def batch_transform(img, trans_func):
    if isinstance(img, Image.Image):
        return trans_func(img)

    assert isinstance(img, np.ndarray)
    assert len(img.shape) == 3

    ch_num = img.shape[2]
    result = []
    for i in range(ch_num):
        tmp_img = Image.fromarray(img[..., i].astype(np.uint8))
        tmp_img = trans_func(tmp_img)
        result.append(np.array(tmp_img))
    return np.dstack(result)


class MultiprocessingDataLoader:
    def __init__(self, num_worker: int):
        self._queue: Queue = Queue(maxsize=20)
        self._dist_queue: Queue = Queue(maxsize=30)
        self._process_pool = []
        for _ in range(num_worker):
            processor = Process(target=self._preprocess_image)
            processor.daemon = True
            self._process_pool.append(processor)
        self._pdist = Process(target=self._distribute_process)
        self._pdist.daemon = True

    def _start_processes(self):
        if not self._pdist.is_alive():
            self._pdist.start()
        for process in self._process_pool:
            if not process.is_alive():
                process.start()

    def _terminate_processes(self):
        self._pdist.terminate()
        for process in self._process_pool:
            process.terminate()


    def _distribute_process(self):
        pass

    def _preprocess_image(self):
        pass


class DataLoader(MultiprocessingDataLoader):
    def __init__(self, feature_files, win_size=256, num_samples=100, min_step_size=0.2, num_worker=4):
        super().__init__(num_worker)
        self.feature_files = feature_files
        random.shuffle(self.feature_files)
        self.win_size = win_size
        self.num_samples = num_samples

        if isinstance(min_step_size, float):
            min_step_size = max(min(abs(min_step_size), 1), 0.01)
            self.min_step_size = round(win_size * min_step_size)
        else:
            self.min_step_size = max(min(abs(min_step_size), win_size), 2)

        self.file_idx = 0

    def _distribute_process(self):
        while True:
            paths = self.feature_files[self.file_idx]
            self._dist_queue.put(paths)
            self.file_idx += 1
            if self.file_idx == len(self.feature_files):
                random.shuffle(self.feature_files)
                self.file_idx = 0

    def _preprocess_image(self):
        while True:
            if not self._queue.full():
                inp_img_path, staff_img_path, symbol_img_path = self._dist_queue.get()

                image, _ = preprocess_image(inp_img_path)

                ratio = random.choice(np.arange(0.2, 1.21, 0.1))
                tar_w = int(ratio * image.size[0])
                tar_h = int(ratio * image.size[1])
                image = imaugs.resize(image, width=tar_w, height=tar_h)
                staff_img = imaugs.resize(staff_img_path, width=tar_w, height=tar_h)
                symbol_img = imaugs.resize(symbol_img_path, width=tar_w, height=tar_h)

                seed = random.randint(0, 1000)
                np.float = float  # Monkey patch to workaround removal of np.float
                perspect_trans = lambda img: imaugs.perspective_transform(img, seed=seed, sigma=70)
                image = np.array(perspect_trans(image))  # RGB image
                staff_img = np.array(perspect_trans(staff_img))  # 1-bit mask
                symbol_img = np.array(perspect_trans(symbol_img))  # 1-bit mask
                staff_img = np.where(staff_img, 1, 0)
                symbol_img = np.where(symbol_img, 1, 0)

                self._queue.put([image, staff_img, symbol_img, ratio])

    def __iter__(self):
        samples = 0

        self._start_processes()

        while samples < self.num_samples:
            image, staff_img, symbol_img, ratio = self._queue.get()

            start_x, start_y = 0, 0
            max_y = image.shape[0] - self.win_size
            max_x = image.shape[1] - self.win_size
            while (start_x < max_x) and (start_y < max_y):
                y_range = range(start_y, start_y+self.win_size)
                x_range = range(start_x, start_x+self.win_size)
                index = np.ix_(y_range, x_range)

                feat = image[index]
                staff = staff_img[index]
                symbol = symbol_img[index]
                neg = np.ones_like(staff) - staff - symbol
                label = np.stack([neg, staff, symbol], axis=-1)

                yield feat, label

                y_step = random.randint(round(self.min_step_size*ratio), round(self.win_size*ratio))
                x_step = random.randint(round(self.min_step_size*ratio), round(self.win_size*ratio))
                start_y = min(start_y + y_step, max_y)
                start_x = min(start_x + x_step, max_x)

        self._terminate_processes()

    def get_dataset(self, batch_size, output_types=None, output_shapes=None):
        def gen_wrapper():
            for data in self:
                yield data

        if output_types is None:
            output_types = (tf.uint8, tf.float32)

        if output_shapes is None:
            output_shapes = ((self.win_size, self.win_size, 3), (self.win_size, self.win_size, 3))

        return tf.data.Dataset.from_generator(
                gen_wrapper, output_types=output_types, output_shapes=output_shapes
            ) \
            .batch(batch_size, drop_remainder=True) \
            .prefetch(tf.data.experimental.AUTOTUNE)


class DsDataLoader(MultiprocessingDataLoader):
    def __init__(self, feature_files, win_size=256, num_samples=100, step_size=0.5, num_worker=4):
        super().__init__(num_worker)
        self.feature_files = feature_files
        random.shuffle(self.feature_files)
        self.win_size = win_size
        self.num_samples = num_samples

        if isinstance(step_size, float):
            step_size = max(abs(step_size), 0.01)
            self.step_size = round(win_size * step_size)
        else:
            self.step_size = max(abs(step_size), 2)

        self.file_idx = 0

    def _distribute_process(self):
        while True:
            paths = self.feature_files[self.file_idx]
            self._dist_queue.put(paths)
            self.file_idx += 1
            if self.file_idx == len(self.feature_files):
                random.shuffle(self.feature_files)
                self.file_idx = 0

    def _preprocess_image(self):
        while True:
            if not self._queue.full():
                inp_img_path, seg_img_path = self._dist_queue.get()

                image, _ = preprocess_image(inp_img_path)
                label = build_label(seg_img_path)

                ratio = random.choice(np.arange(0.2, 1.21, 0.1))
                tar_w = int(ratio * image.size[0])
                tar_h = int(ratio * image.size[1])
                trans_func = lambda img: imaugs.resize(img, width=tar_w, height=tar_h)
                image = batch_transform(image, trans_func)
                label = batch_transform(label, trans_func)

                seed = random.randint(0, 1000)
                np.float = float  # Monkey patch to workaround removal of np.float
                perspect_trans = lambda img: imaugs.perspective_transform(img, seed=seed, sigma=70)
                image = np.array(batch_transform(image, perspect_trans))  # RGB image
                label = np.array(batch_transform(label, perspect_trans))

                self._queue.put([image, label, ratio])

    def __iter__(self):
        samples = 0

        self._start_processes()

        while samples < self.num_samples:
            image, label, ratio = self._queue.get()

            staff = label[..., 1]
            yidx, _ = np.where(staff>0)
            if len(yidx) > 0:
                max_y = min(np.max(yidx) + 100, image.shape[0])
            else:
                max_y = image.shape[0]

            max_y = max_y - self.win_size
            max_x = image.shape[1] - self.win_size
            grid_x = range(0, max_x, round(self.step_size*ratio))
            grid_y = range(0, max_y, round(self.step_size*ratio))
            meshgrid = np.meshgrid(grid_x, grid_y, indexing='ij')
            coords = np.dstack(meshgrid).reshape(-1, 2)
            random.shuffle(coords)
            for start_x, start_y in coords:
                y_range = range(start_y, start_y+self.win_size)
                x_range = range(start_x, start_x+self.win_size)
                index = np.ix_(y_range, x_range)

                feat = image[index]
                ll = label[index]
                yield feat, ll

        self._terminate_processes()
    def get_dataset(self, batch_size, output_types=None, output_shapes=None):
        def gen_wrapper():
            for data in self:
                yield data

        if output_types is None:
            output_types = (tf.uint8, tf.float32)

        if output_shapes is None:
            output_shapes = ((self.win_size, self.win_size, 3), (self.win_size, self.win_size, CHANNEL_NUM))

        return tf.data.Dataset.from_generator(
                gen_wrapper, output_types=output_types, output_shapes=output_shapes
            ) \
            .batch(batch_size, drop_remainder=True) \
            .prefetch(tf.data.experimental.AUTOTUNE)


def lr_scheduler(epoch, lr, update_after=5, dec_every=3, dec_rate=0.5):
    if epoch >= update_after and (epoch - update_after) % dec_every == 0:
        lr *= dec_rate
    return max(lr, 5e-8)


class WarmUpLearningRate(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, init_lr=0.1, warm_up_steps=1000, decay_step=3000, decay_rate=0.25, min_lr=1e-8):
        self.init_lr = init_lr
        self.warm_up_steps = warm_up_steps
        self.decay_step = decay_step
        self.decay_rate = decay_rate
        self.min_lr = min_lr

        self.warm_step_size = (init_lr - min_lr) / warm_up_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm_lr = self.min_lr + self.warm_step_size * step

        offset = step - self.warm_up_steps
        cycle = offset // self.decay_step
        start_lr = self.init_lr * tf.pow(self.decay_rate, cycle)
        end_lr = start_lr * self.decay_rate
        step_size = (start_lr - end_lr) / self.decay_step
        lr = start_lr - (offset - cycle * self.decay_step) * step_size
        true_lr = tf.where(offset > 0, lr, warm_lr)
        return tf.maximum(true_lr, self.min_lr)

    def get_config(self):
        return {"init_lr": self.init_lr,
                "warm_up_steps": self.warm_up_steps,
                "decay_step": self.decay_step,
                "decay_rate": self.decay_rate,
                "min_lr": self.min_lr}

def sigmoid_focal_crossentropy_tf(y_true, y_pred, alpha: float = 0.25, gamma: float = 2.0, from_logits: bool = False,):
    """
    Args:
        y_true: nhãn thực, shape (B, H, W, C)
        y_pred: dự đoán, shape (B, H, W, C)
        alpha : balancing factor, mặc định 0.25
        gamma : modulating factor, mặc định 2.0
        from_logits: True nếu y_pred chưa qua sigmoid/softmax
    Returns:
        loss tensor shape (B, H, W) — reduce_sum theo axis=-1
    """
    y_pred = tf.convert_to_tensor(y_pred)
    y_true = tf.cast(y_true, dtype=y_pred.dtype)

    ce = tf.keras.backend.binary_crossentropy(y_true, y_pred, from_logits=from_logits)

    # Xác suất của class đúng
    if from_logits:
        pred_prob = tf.sigmoid(y_pred)
    else:
        pred_prob = y_pred

    p_t = (y_true * pred_prob) + ((1 - y_true) * (1 - pred_prob))

    # Alpha weighting: foreground được boost, background bị giảm
    alpha = tf.cast(alpha, dtype=y_true.dtype)
    alpha_factor = y_true * alpha + (1 - y_true) * (1 - alpha)

    # Focal modulating: giảm loss ở pixel dễ, tập trung pixel khó
    gamma = tf.cast(gamma, dtype=y_true.dtype)
    modulating_factor = tf.pow((1.0 - p_t), gamma)

    # reduce_sum theo axis=-1 (channel) giống TFA — shape (B, H, W)
    return tf.reduce_sum(alpha_factor * modulating_factor * ce, axis=-1)


def train_model(
    dataset_path,
    train_val_split=0.1,
    learning_rate=5e-4,
    epochs=15,
    steps=1000,
    batch_size=2,
    val_steps=200,
    val_batch_size=2,
    early_stop=8,
    data_model="segnet"):
    if data_model == "segnet":
        feat_files = get_deep_score_data_paths(dataset_path)
    else:
        feat_files = get_cvc_data_paths(dataset_path)
    random.shuffle(feat_files)
    split_idx = round(train_val_split * len(feat_files))
    train_files = feat_files[split_idx:]
    val_files = feat_files[:split_idx]

    print(f"Loading dataset. Train/validation: {len(train_files)}/{len(val_files)}")
    if data_model == "segnet":
        win_size=288
        train_data = DsDataLoader(train_files, win_size=win_size, num_samples=epochs*steps*batch_size) \
            .get_dataset(batch_size)
        val_data = DsDataLoader(val_files, win_size=win_size, num_samples=epochs*val_steps*val_batch_size) \
            .get_dataset(val_batch_size)
        model = u_net(win_size=win_size, out_class=CHANNEL_NUM)
    else:
        win_size=256
        train_data = DataLoader(train_files, win_size=win_size, num_samples=epochs*steps*batch_size) \
            .get_dataset(batch_size)
        val_data = DataLoader(val_files, win_size=win_size, num_samples=epochs*val_steps*val_batch_size) \
            .get_dataset(val_batch_size)
        model = semantic_segmentation(win_size=256, out_class=3)

    print("Initializing model")
    optim = tf.keras.optimizers.Adam(learning_rate=WarmUpLearningRate(learning_rate))

    # losses.SigmoidFocalCrossEntropy
    # alpha=0.25: downweight background
    # gamma=2.0 : tập trung vào pixel khó
    loss = lambda y_true, y_pred: tf.reduce_mean(sigmoid_focal_crossentropy_tf(y_true, y_pred, alpha=0.25, gamma=2.0))
    model.compile(optimizer=optim, loss=loss, metrics=['accuracy'])

    ckpt_dir = "/kaggle/working/checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, "seg_unet.keras")

    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=early_stop, monitor="val_accuracy"),
        tf.keras.callbacks.ModelCheckpoint(filepath=ckpt_path, save_weights_only=False, monitor="val_accuracy")]

    print("Start training")
    try:
        model.fit(
            train_data,
            validation_data=val_data,
            epochs=epochs,
            steps_per_epoch=steps,
            validation_steps=val_steps,
            callbacks=callbacks)
        return model
    except Exception as e:
        print(e)
        return model

In [38]:
DS2_DENSE_PATH = "/kaggle/input/datasets/nguyncaonam/ds2-dense/ds2_dense"
CVC_PATH = "/kaggle/input/datasets/nguyncaonam/cvcmusicma-sr/CvcMuscima-Distortions"
CKPT_PATH = "/kaggle/working/checkpoints/seg_unet.keras" # Load CheckPoints

OUTPUT_ROOT = "/kaggle/working/models"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

def write_text_to_file(text, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def get_model_base_name(model_name: str) -> str:
    timestamp = str(round(time.time()))
    return f"{model_name}_{timestamp}"


def prepare_classifier_data():
    # if not os.path.exists("train_data"):
        collect_data(2000)


model_type = "segnet"     # "segnet", "unet", "unet_from_checkpoint", "segnet_from_checkpoint", ...


def make_outdir(name: str) -> str:
    os.makedirs(name, exist_ok=True)
    return name


if model_type == "segnet":
    model = train_model(DS2_DENSE_PATH, data_model=model_type, steps=1500, epochs=15)
    filename = make_outdir(os.path.join(OUTPUT_ROOT, get_model_base_name(model_type)))
    write_text_to_file(model.to_json(), os.path.join(filename, "arch.json"))
    model.save_weights(os.path.join(filename, "weights.weights.h5"))

elif model_type == "unet":
    model = train_model(CVC_PATH, data_model=model_type, steps=1500, epochs=15)
    filename = make_outdir(os.path.join(OUTPUT_ROOT, get_model_base_name(model_type)))
    write_text_to_file(model.to_json(), os.path.join(filename, "arch.json"))
    model.save_weights(os.path.join(filename, "weights.weights.h5"))

elif model_type == "unet_from_checkpoint" or model_type == "segnet_from_checkpoint":
    model = tf.keras.models.load_model(CKPT_PATH, custom_objects={"WarmUpLearningRate": WarmUpLearningRate})
    filename = make_outdir(os.path.join(OUTPUT_ROOT, get_model_base_name(model_type.split("_")[0])))
    write_text_to_file(model.to_json(), os.path.join(filename, "arch.json"))
    model.save_weights(os.path.join(filename, "weights.weights.h5"))

elif model_type == "rests_above8":
    prepare_classifier_data()
    train_rests_above8(get_model_base_name(model_type))

elif model_type == "rests":
    prepare_classifier_data()
    train_rests(get_model_base_name(model_type))

elif model_type == "all_rests":
    prepare_classifier_data()
    train_all_rests(get_model_base_name(model_type))

elif model_type == "sfn":
    prepare_classifier_data()
    train_sfn(get_model_base_name(model_type))

elif model_type == "clef":
    prepare_classifier_data()
    train_clefs(get_model_base_name(model_type))

else:
    print("Unknown model: " + model_type)
    sys.exit(1)

Loading dataset. Train/validation: 1543/171
Initializing model
Start training
Epoch 1/15


2026-04-02 03:09:44.357388: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-02 03:09:44.660769: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-02 03:09:51.026896: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-02 03:09:51.257130: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-02 03:09:52.381657: E external/local_xla/xla/stream_

1500/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.8822 - loss: 0.0549

/usr/local/lib/python3.12/dist-packages/keras/src/saving/serialization_lib.py:390: UserWarning: The object being serialized includes a `lambda`. This is unsafe. In order to reload the object, you will have to pass `safe_mode=False` to the loading function. Please avoid using `lambda` in the future, and use named Python functions instead. This is the `lambda` being serialized:     loss = lambda y_true, y_pred: tf.reduce_mean(sigmoid_focal_crossentropy_tf(y_true, y_pred, alpha=0.25, gamma=2.0))

  return {key: serialize_keras_object(value) for key, value in obj.items()}


1500/1500 ━━━━━━━━━━━━━━━━━━━━ 534s 276ms/step - accuracy: 0.8822 - loss: 0.0549 - val_accuracy: 0.9734 - val_loss: 0.0135
Epoch 2/15
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 396s 264ms/step - accuracy: 0.9412 - loss: 0.0123 - val_accuracy: 0.9214 - val_loss: 0.0285
Epoch 3/15
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 396s 264ms/step - accuracy: 0.9705 - loss: 0.0085 - val_accuracy: 0.9016 - val_loss: 0.0300
Epoch 4/15
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 441s 294ms/step - accuracy: 0.9628 - loss: 0.0096 - val_accuracy: 0.9823 - val_loss: 0.0125
Epoch 5/15
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 394s 263ms/step - accuracy: 0.9544 - loss: 0.0093 - val_accuracy: 0.9655 - val_loss: 0.0182
Epoch 6/15
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 394s 263ms/step - accuracy: 0.9625 - loss: 0.0094 - val_accuracy: 0.9554 - val_loss: 0.0246
Epoch 7/15
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 392s 262ms/step - accuracy: 0.9614 - loss: 0.0095 - val_accuracy: 0.9494 - val_loss: 0.0121
Epoch 8/15
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 442s 295ms/step - accuracy: 0.9

In [39]:
# !zip -r /kaggle/working/checkpoints.zip /kaggle/working/checkpoints